# 02 — Dataset Curation and Annotation Pipeline

This notebook implements an automated, leak-free dataset validation and inspection pipeline for YOLOv8 object detection projects.

---

## 1. The YOLO Dataset Specification

Ultralytics YOLO requires a strict parallel directory layout where images and labels mirror each other:

```text
dataset_root/
├── data.yaml
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

Every image `<stem>.<ext>` must correspond to a label text file `<stem>.txt`. An image containing no objects must have an empty `.txt` file or omit the file depending on the negative sample configuration.

In [ ]:
from pathlib import Path

import yaml
from PIL import Image, ImageDraw

DATASET_ROOT = Path("../data/human_detection_mini")
assert DATASET_ROOT.exists(), f"Dataset directory not found at {DATASET_ROOT}"

print(f"Dataset root verified: {DATASET_ROOT.resolve()}")

## 2. Parsing and Validating YOLO Annotation Files

Each line in a YOLO annotation file specifies:
$$\langle \text{class\_id} \rangle \quad \langle x_c \rangle \quad \langle y_c \rangle \quad \langle w \rangle \quad \langle h \rangle$$

Validation rules:
1. Exactly 5 tokens per line.
2. $\text{class\_id} \in \mathbb{Z}_{\ge 0}$ and $\text{class\_id} < nc$.
3. All coordinates must lie strictly within $[0, 1]$.
4. Extrapolated bounding box corners must satisfy $x_c - w/2 \ge 0$ and $x_c + w/2 \le 1$.

In [ ]:
from dataclasses import dataclass


@dataclass
class YOLOAnnotation:
    class_id: int
    xc: float
    yc: float
    w: float
    h: float

    def to_voc(self, img_w: int, img_h: int) -> tuple[float, float, float, float]:
        xmin = (self.xc - self.w / 2.0) * img_w
        ymin = (self.yc - self.h / 2.0) * img_h
        xmax = (self.xc + self.w / 2.0) * img_w
        ymax = (self.yc + self.h / 2.0) * img_h
        return xmin, ymin, xmax, ymax


def parse_and_validate_annotation(filepath: Path, num_classes: int = 1) -> list[YOLOAnnotation]:
    """Parse and validate a YOLO format .txt label file."""
    annotations = []
    if not filepath.exists() or filepath.stat().st_size == 0:
        return annotations

    with filepath.open(encoding="utf-8") as f:
        for line_no, raw_line in enumerate(f, start=1):
            line = raw_line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f"{filepath}:{line_no} - Expected 5 values, got {len(parts)}")

            cls_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])

            if not (0 <= cls_id < num_classes):
                raise ValueError(
                    f"{filepath}:{line_no} - Invalid class_id {cls_id} (nc={num_classes})"
                )

            for val, name in [(xc, "xc"), (yc, "yc"), (w, "w"), (h, "h")]:
                if not (0.0 <= val <= 1.0):
                    raise ValueError(
                        f"{filepath}:{line_no} - Coordinate {name}={val} out of bounds [0, 1]"
                    )

            annotations.append(YOLOAnnotation(cls_id, xc, yc, w, h))

    return annotations


# Audit validation labels
val_labels_dir = DATASET_ROOT / "labels" / "val"
total_annotations = 0
for txt_file in sorted(val_labels_dir.glob("*.txt")):
    annos = parse_and_validate_annotation(txt_file, num_classes=1)
    total_annotations += len(annos)
    print(f"Validated {txt_file.name}: {len(annos)} targets")

assert total_annotations > 0
print(f"Validation complete: All {total_annotations} targets passed schema checks.")

## 3. Dataset Configuration Generation (`data.yaml`)

Ultralytics YOLO consumes a YAML configuration pointing to dataset splits and mapping class indices to names.

In [ ]:
def create_dataset_yaml(
    output_path: Path,
    dataset_path: str,
    train_rel: str,
    val_rel: str,
    class_names: list[str],
) -> None:
    """Generate a compliant data.yaml file."""
    config = {
        "path": dataset_path,
        "train": train_rel,
        "val": val_rel,
        "test": None,
        "nc": len(class_names),
        "names": dict(enumerate(class_names)),
    }
    with output_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)


# Test reading existing yaml
yaml_file = DATASET_ROOT / "data.yaml"
with yaml_file.open(encoding="utf-8") as f:
    loaded_config = yaml.safe_load(f)

print("Loaded data.yaml configuration:")
print(yaml.dump(loaded_config, sort_keys=False))
assert loaded_config["nc"] == 1
assert loaded_config["names"][0] == "human"

## 4. Visual Inspection: Ground-Truth Bounding Box Rendering

Visual inspection ensures bounding boxes match physical entities accurately before beginning compute-intensive training.

In [ ]:
def render_annotations_on_image(
    image_path: Path, label_path: Path, class_names: dict[int, str]
) -> Image.Image:
    """Load an image and render bounding boxes and class tags."""
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    annos = parse_and_validate_annotation(label_path, num_classes=len(class_names))

    w, h = img.size
    for anno in annos:
        xmin, ymin, xmax, ymax = anno.to_voc(w, h)
        label_text = f"{class_names.get(anno.class_id, 'object')}"

        # Draw box
        draw.rectangle([xmin, ymin, xmax, ymax], outline=(46, 204, 113), width=3)
        # Draw tag
        draw.rectangle([xmin, ymin - 18, xmin + 65, ymin], fill=(46, 204, 113))
        draw.text((xmin + 4, ymin - 16), label_text, fill=(255, 255, 255))

    return img


# Inspect human_val_01
sample_img = DATASET_ROOT / "images" / "val" / "human_val_01.jpg"
sample_lbl = DATASET_ROOT / "labels" / "val" / "human_val_01.txt"
rendered = render_annotations_on_image(sample_img, sample_lbl, loaded_config["names"])

print(f"Rendered image size: {rendered.size}")
assert rendered.size == (640, 640)
print("Bounding box overlay rendered successfully.")